# Variational Autoencoder (VAE) - PyTorch

**Goal:** Generate MNIST-like digits from a probabilistic latent space.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** A probabilistic latent space supports reconstruction and sampling.
- **Where it is used:** generative modeling, latent-space exploration, and controllable sampling.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Variational Autoencoder: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['input', 'mu/sigma', 'sample']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.sin(x)*np.exp(-.15*x**2)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.55,.25,.20])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['recon', 'KL', 'noise'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
from sklearn.datasets import load_digits

X = (load_digits().data / 16.0).astype("float32")
train_loader = DataLoader(TensorDataset(torch.tensor(X)), batch_size=64, shuffle=True)


In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim: int = 8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.mu = nn.Linear(32, latent_dim)
        self.logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid())

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        hidden = self.encoder(x)
        mu, logvar = self.mu(hidden), self.logvar(hidden)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


def vae_loss(reconstruction, x, mu, logvar):
    recon = nn.functional.binary_cross_entropy(reconstruction, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon + kl) / x.size(0)


model = VAE().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)


In [ ]:
for epoch in range(20):
    model.train()
    losses = []
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f"epoch={epoch+1:02d} loss={np.mean(losses):.3f}")

with torch.no_grad():
    samples = model.decoder(torch.randn(8, 8, device=device)).cpu().reshape(-1, 8, 8)
print("Generated sample batch shape:", tuple(samples.shape))
